In [38]:
# Import Library yang dibutuhkan
import pickle # menggunakan pickle untuk load data
import sys # menggunakan sys untuk mengatasi error pada versi pandas terbaru
import pandas as pd
import numpy as np
df_sample= pd.read_pickle('../data/wm811k_sample_v2_stratified.pkl')
print("Library dan data yang dibutuhkan sudah diimport")

Library dan data yang dibutuhkan sudah diimport


In [39]:
df_sample.isnull().sum()

waferMap                0
dieSize                 0
lotName                 0
waferIndex              0
trianTestLabel          0
failureType             0
failureType_clean       0
trainTestLabel_clean    0
stratify_key            0
dtype: int64

In [40]:
# apakah ada duplikasi baris pada data?
df_sample.duplicated(subset=['lotName', 'waferIndex', 'dieSize']).sum()

np.int64(0)

In [41]:
print(f"berikut column pada dataset: {df.columns}")
print(f"tipe data pada dataset: {df.dtypes}")

berikut column pada dataset: Index(['waferMap', 'dieSize', 'lotName', 'waferIndex', 'trianTestLabel',
       'failureType', 'failureType_clean'],
      dtype='str')
tipe data pada dataset: waferMap              object
dieSize              float64
lotName               object
waferIndex           float64
trianTestLabel        object
failureType           object
failureType_clean        str
dtype: object


In [57]:
pd.crosstab(df_sample['trainTestLabel_clean'], df_sample['failureType_clean'])

failureType_clean,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none
trainTestLabel_clean,,,,,,,,,
Test,832,146,2772,1126,1973,95,257,693,110701
Training,3462,409,2417,8554,1620,54,609,500,36730
none,0,0,0,0,0,0,0,0,638507


In [ ]:
# mari kita liat berapa banyak memori yang digunakan oleh dataset sebelum optimasi
memori_sebelum = df_sample.memory_usage(deep=True).sum() / 1e6
print(f"Memori sebelum optimasi: {memori_sebelum:.2f} MB")

Memori sebelum optimasi: 553.19 MB


In [55]:
# mari kita optimasi tipe data pada kolom kategorikal agar lebih hemat memori
kolom_kategorikal = ['stratify_key', 'failureType_clean', 'trainTestLabel_clean', 'lotName']
# kita buat fungsi untuk mengubah tipe data kolom kategorikal menjadi 'category'
def optimize_categorical_columns(df, columns):
    for col in columns:
        df[col] = df[col].astype('category')
    return df

df_sample = optimize_categorical_columns(df_sample, kolom_kategorikal)

# kita akan downcast kolom numerik agar lebih hemat memori
# kita cek apakah kolom dieSize dan waferIndex sudah integer
print((df_sample['dieSize'] % 1 != 0).sum())      # harus 0
print((df_sample['waferIndex'] % 1 != 0).sum())   # harus 0

# mari kita downcast kolom dieSize dan waferIndex menjadi integer
df_sample['dieSize'] = pd.to_numeric(df_sample['dieSize'], downcast='integer')
df_sample['waferIndex'] = pd.to_numeric(df_sample['waferIndex'], downcast='integer')

# kita cek apakah sudah teroptimasi dengan baik
memori_sesudah = df_sample.memory_usage(deep=True).sum() / 1e6
print(f"Memori sesudah optimasi: {memori_sesudah:.2f} MB")
print(f"Penghematan: {(1 - memori_sesudah/memori_sebelum)*100:.1f}%")

0
0
Memori sesudah optimasi: 344.55 MB
Penghematan: 37.7%


In [56]:
# mari kita ubah label kolom menjadi lebih mudah dibaca
kode_singkat = {
    'unlabeled': 'UNL',
    'reviewed_no_defect': 'RND',
    'Center': 'C',
    'Donut': 'D',
    'Edge-Loc': 'EL',
    'Edge-Ring': 'ER',
    'Loc': 'L',
    'Near-full': 'NF',
    'Random': 'R',
    'Scratch': 'S'
}
# kita akan membuat kolom baru 'stratify_key_short' yang berisi kode singkat dari 'stratify_key'
# dengan apply() dan lambda function akan mengubah setiap value pada kolom 'stratify_key' menjadi kode singkat sesuai dengan dictionary kode_singkat
df_sample['stratify_key_short'] = df_sample['stratify_key'].apply(lambda x: kode_singkat[x])

# mari kita cek apakah ada nilai null pada kolom 'stratify_key_short'
print(df_sample['stratify_key_short'].isnull().sum())  # harus 0

0


In [59]:
# mari kita liat sebaran data
print(df_sample['dieSize'].describe())
print(df_sample['dieSize'].quantile([0.25, 0.5, 0.75]))

count    811457.000000
mean       1840.998585
std        2254.987374
min           3.000000
25%         710.000000
50%         953.000000
75%        1902.000000
max       48099.000000
Name: dieSize, dtype: float64
0.25     710.0
0.50     953.0
0.75    1902.0
Name: dieSize, dtype: float64
